In [139]:
!pip install netron
!pip install onnx onnxscript

In [140]:
import torch
import numpy as np

def transformar_hex(file_path):
  labels = []
  image_list = []

  with open(file_path, 'r') as f:
    for i, linea in enumerate(f):
      if linea[:6] == "DATA_S":
        label = linea.strip().split(' ')
        labels.append(int(label[1]))
      elif linea[:6] == "DATA_E":
        pass
      else:
        imagen_hex = linea.strip().split(',')
        pixel_values = [int(h, 16) / 255.0 for h in imagen_hex if h.strip()]
        if len(pixel_values) < 9216:
          diferencia = 9216 - len(pixel_values)
          pixel_values.extend([0.0] * diferencia)
        image_list.append(pixel_values)

  datos = np.array(image_list)
  x = datos.reshape(-1, 96, 96, 1)
  y = np.array(labels)
  return x,y

In [141]:
x,y = transformar_hex('DATA_EMBUT.txt')
print(len(y))

410


In [148]:
import tensorflow as tf
from tensorflow.keras import layers, models

def modelo_keras(input_shape, num_classes):
    model = models.Sequential([

        layers.Conv2D(32, (3, 3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),

        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])

    return model

topologia_keras = modelo_keras(input_shape=(96, 96, 1), num_classes=2)
topologia_keras.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

topologia_keras.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_18 (Conv2D)              │ (None, 96, 96, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 96, 96, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 96, 96, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 48, 48, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 48, 48, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 48, 48, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 24, 24, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │     1,179,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,273,410 (4.86 MB)

 Trainable params: 1,272,962 (4.86 MB)

 Non-trainable params: 448 (1.75 KB)

In [149]:
history = topologia_keras.fit(
    x,
    y,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    shuffle=True
)

Epoch 1/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 8s 131ms/step - accuracy: 0.5549 - loss: 3.5600 - val_accuracy: 0.9024 - val_loss: 0.2988
Epoch 2/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 5s 132ms/step - accuracy: 0.5671 - loss: 3.8818 - val_accuracy: 0.9146 - val_loss: 1.1270
Epoch 3/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 5s 130ms/step - accuracy: 0.5793 - loss: 2.3623 - val_accuracy: 0.9146 - val_loss: 0.5781
Epoch 4/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 10s 136ms/step - accuracy: 0.6585 - loss: 0.7413 - val_accuracy: 0.9146 - val_loss: 0.3574
Epoch 5/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 5s 126ms/step - accuracy: 0.7348 - loss: 0.5542 - val_accuracy: 0.9146 - val_loss: 0.2991
Epoch 6/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 10s 125ms/step - accuracy: 0.7043 - loss: 0.6422 - val_accuracy: 0.9146 - val_loss: 0.3526
Epoch 7/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 6s 146ms/step - accuracy: 0.7713 - loss: 0.5110 - val_accuracy: 0.9146 - val_loss: 0.4717
Epoch 8/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 5s 126ms/step - accuracy: 0.8720 - loss: 0.3234 - val_accuracy: 

In [150]:
import torch
import torch.nn as nn

class BinaryClassifier(nn.Module):
    def __init__(self, topology):
        super(BinaryClassifier, self).__init__()

        match topology:

        # MLP Simple
          case "simpleMLP":
              self.model = nn.Sequential(
                  nn.Flatten(),
                  nn.Linear(96*96, 512),
                  nn.ReLU(),
                  nn.Linear(512, 128),
                  nn.ReLU(),
                  nn.Linear(128, 1)
              )

        # CNN Ligera
          case "lightCNN":
              self.model = nn.Sequential(
                  nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
                  nn.MaxPool2d(2),
                  nn.Flatten(),
                  nn.Linear(16 * 48 * 48, 1)
              )

        # CNN Profunda
          case "deepCNN":
              self.model = nn.Sequential(
                  nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                  nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                  nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                  nn.Flatten(),
                  nn.Linear(64 * 12 * 12, 1)
              )

        # CNN con Dropout
          case "dropoutCNN":
              self.model = nn.Sequential(
                  nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                  nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                  nn.Flatten(),
                  nn.Dropout(0.5),
                  nn.Linear(32 * 24 * 24, 1)
              )

    def forward(self, x):
        return self.model(x)

In [151]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


X_tensor = torch.tensor(x, dtype=torch.float32).permute(0, 3, 1, 2)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=10, shuffle=True)

In [152]:
def entrenar_modelo(modelo, dataloader, epochs=20, lr=0.001):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(modelo.parameters(), lr=lr)

    modelo.train()

    accuracy = 0
    for epoch in range(epochs):
        running_loss = 0.0
        correctas = 0
        total = 0
        for inputs, labels in dataloader:
            optimizer.zero_grad()
            outputs = modelo(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            probs = torch.sigmoid(outputs)

            predicciones = (probs > 0.5).float()


            correctas += (predicciones == labels).sum().item()
            total += labels.size(0)

        epoch_acc = (correctas / total) * 100
        accuracy = epoch_acc

        print(f"Época {epoch+1}/{epochs} - Loss: {running_loss/len(dataloader):.4f} - Accuracy: {epoch_acc:.2f}%")

    return accuracy

In [154]:
topologia_simple = BinaryClassifier(topology="simpleMLP")
topologia_cnn_simple = BinaryClassifier(topology="lightCNN")
topologia_cnn_profunda = BinaryClassifier(topology="deepCNN")
topologia_cnn_dropout = BinaryClassifier(topology="dropoutCNN")

print(topologia_simple)

acc_simple = entrenar_modelo(topologia_simple, dataloader)
acc_cnn_s = entrenar_modelo(topologia_cnn_simple, dataloader)
acc_cnn_p = entrenar_modelo(topologia_cnn_profunda, dataloader)
acc_cnn_d = entrenar_modelo(topologia_cnn_dropout, dataloader)

BinaryClassifier(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=9216, out_features=512, bias=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=1, bias=True)
  )
)
Época 1/20 - Loss: 0.7731 - Accuracy: 54.15%
Época 2/20 - Loss: 0.7500 - Accuracy: 51.71%
Época 3/20 - Loss: 0.6901 - Accuracy: 51.71%
Época 4/20 - Loss: 0.6925 - Accuracy: 52.93%
Época 5/20 - Loss: 0.6846 - Accuracy: 55.12%
Época 6/20 - Loss: 0.6933 - Accuracy: 56.59%
Época 7/20 - Loss: 0.6780 - Accuracy: 57.32%
Época 8/20 - Loss: 0.6843 - Accuracy: 55.85%
Época 9/20 - Loss: 0.6744 - Accuracy: 58.54%
Época 10/20 - Loss: 0.6863 - Accuracy: 55.85%
Época 11/20 - Loss: 0.6704 - Accuracy: 57.80%
Época 12/20 - Loss: 0.6723 - Accuracy: 56.34%
Época 13/20 - Loss: 0.6484 - Accuracy: 62.93%
Época 14/20 - Loss: 0.5901 - Accuracy: 69.51%
Época 15/20 - Loss: 0.6337 - Accuracy: 62.20%
Época 16/20 -

In [155]:
print("MPL Acc:",acc_simple)
print("CNN Simple Acc:", acc_cnn_s)
print("CNN Profundo Acc:", acc_cnn_p)
print("Cnn Dropout Acc:", acc_cnn_d)

MPL Acc: 77.3170731707317
CNN Simple Acc: 99.51219512195122
CNN Profundo Acc: 99.51219512195122
Cnn Dropout Acc: 96.34146341463415


In [135]:
from sklearn.metrics import f1_score, accuracy_score

def evaluar_f1(modelo, dataloader):
    modelo.eval()
    y_reales = []
    y_preds = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            logits = modelo(inputs)
            preds = (logits > 0).float()

            y_reales.extend(labels.numpy())
            y_preds.extend(preds.numpy())

    score = f1_score(y_reales, y_preds)
    acc = accuracy_score(y_reales, y_preds)
    print(f"F1-Score Final: {score:.4f}")
    print(f"Accuracy Final: {acc:.4f}")
    return score

evaluar_f1(topologia_cnn_dropout, dataloader)

F1-Score Final: 0.9781
Accuracy Final: 0.9780


0.9781021897810219

In [173]:
import torch.onnx

topologia_cnn_dropout.eval()
topologia_cnn_profunda.eval()
topologia_cnn_simple.eval()
topologia_simple.eval()

def exportar_modelos(modelo,name):
  dummy_input = torch.randn(1, 1, 96, 96)
  try:
      torch.onnx.export(
          modelo,
          dummy_input,
          f"{name}.onnx",
          export_params=True,
          opset_version=14,
          input_names=['entrada_hex'],
          output_names=['prediccion']
      )
      print("Trivago")
  except Exception as e:
      print(f"Error: {e}")

In [174]:
exportar_modelos(topologia_simple,"mlp")
exportar_modelos(topologia_cnn_simple,"cnn_simple")
exportar_modelos(topologia_cnn_profunda,"cnn_profunda")
exportar_modelos(topologia_cnn_dropout,"cnn_dropout")

W0506 01:06:13.786000 6870 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `BinaryClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BinaryClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
W0506 01:06:14.605000 6870 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Trivago
[torch.onnx] Obtain model graph for `BinaryClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BinaryClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
W0506 01:06:15.377000 6870 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Trivago
[torch.onnx] Obtain model graph for `BinaryClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BinaryClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
W0506 01:06:16.185000 6870 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Trivago
[torch.onnx] Obtain model graph for `BinaryClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BinaryClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Trivago


In [163]:
topologia_keras.save('mi_modelo_keras.h5')

In [184]:
import netron
import IPython
from google.colab import output

try:
  netron.stop()
  model_path = "cnn_dropout.onnx"
  netron.start(model_path, 8070)
  output.serve_kernel_port_as_iframe(8070)
except Exception as e:
  print(f"Error: {e}")


<IPython.core.display.Javascript object>

In [185]:
netron.stop()